In [1]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset
from rich.console import Console
from rich.tree import Tree
import time

# Check if your GTX 1650 is detected
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


TASK 1 :Model Architecture Parsing

In [2]:
def parse_model_to_tree(module: nn.Module, tree: Tree, depth: int = 0, max_depth: int = 4):
    """
    Recursively parses a PyTorch module and builds a rich visual tree.
    """
    if depth > max_depth:
        return

    for name, child in module.named_children():
        # Get parameter count for this specific block
        params = sum(p.numel() for p in child.parameters())
        param_str = f" [cyan]({params:,} params)[/cyan]" if params > 0 else ""
        
        # Create a clean label
        node_label = f"[bold green]{name}[/bold green]: [yellow]{child.__class__.__name__}[/yellow]{param_str}"
        branch = tree.add(node_label)
        
        # Recurse into children
        parse_model_to_tree(child, branch, depth + 1, max_depth)

print("Loading models for parsing (This might take a minute if downloading TinyLlama)...")
gpt2_model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")
tiny_llama = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

console = Console()

# 1. Visualize GPT-2 Architecture
root_tree_gpt2 = Tree(f"[bold blue]GPT-2 Architecture (Total Params: {sum(p.numel() for p in gpt2_model.parameters()):,})[/bold blue]")
parse_model_to_tree(gpt2_model, root_tree_gpt2, max_depth=3)
console.print(root_tree_gpt2)

print("\n" + "="*60 + "\n")

# 2. Visualize TinyLlama Architecture
root_tree_llama = Tree(f"[bold magenta]TinyLlama Architecture (Total Params: {sum(p.numel() for p in tiny_llama.parameters()):,})[/bold magenta]")
parse_model_to_tree(tiny_llama, root_tree_llama, max_depth=3)
console.print(root_tree_llama)

Loading models for parsing (This might take a minute if downloading TinyLlama)...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

c:\Users\Hardik\Desktop\SE\llm-systems-assessment\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Hardik\.cache\huggingface\hub\models--openai-community--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 Architecture (Total Params: 124,439,808)
├── transformer: GPT2Model (124,439,808 params)
│   ├── wte: Embedding (38,597,376 params)
│   ├── wpe: Embedding (786,432 params)
│   ├── drop: Dropout
│   ├── h: ModuleList (85,054,464 params)
│   │   ├── 0: GPT2Block (7,087,872 params)
│   │   │   ├── ln_1: LayerNorm (1,536 params)
│   │   │   ├── attn: GPT2Attention (2,362,368 params)
│   │   │   ├── ln_2: LayerNorm (1,536 params)
│   │   │   └── mlp: GPT2MLP (4,722,432 params)
│   │   ├── 1: GPT2Block (7,087,872 params)
│   │   │   ├── ln_1: LayerNorm (1,536 params)
│   │   │   ├── attn: GPT2Attention (2,362,368 params)
│   │   │   ├── ln_2: LayerNorm (1,536 params)
│   │   │   └── mlp: GPT2MLP (4,722,432 params)
│   │   ├── 2: GPT2Block (7,087,872 params)
│   │   │   ├── ln_1: LayerNorm (1,536 params)
│   │   │   ├── attn: GPT2Attention (2,362,368 params)
│   │   │   ├── ln_2: LayerNorm (1,536 params)
│   │   │   └── mlp: GPT2MLP (4,722,432 params)
│   │   ├── 3: GPT2Block (7,087,872 params)
│   │   │   ├── ln_1: LayerNorm (1,536 params)
│   │   │   ├── attn: GPT2Attention (2,362,368 params)
│   │   │   ├── ln_2: LayerNorm (1,536 params)
│   │   │   └── mlp: GPT2MLP (4,722,432 params)
│   │   ├── 4: GPT2Block (7,087,872 params)
│   │   │   ├── ln_1: LayerNorm (1,536 params)
│   │   │   ├── attn: GPT2Attention (2,362,368 params)
│   │   │   ├── ln_2: LayerNorm (1,536 params)
│   │   │   └── mlp: GPT2MLP (4,722,432 params)
│   │   ├── 5: GPT2Block (7,087,872 params)
│   │   │   ├── ln_1: LayerNorm (1,536 params)
│   │   │   ├── attn: GPT2Attention (2,362,368 params)
│   │   │   ├── ln_2: LayerNorm (1,536 params)
│   │   │   └── mlp: GPT2MLP (4,722,432 params)
│   │   ├── 6: GPT2Block (7,087,872 params)
│   │   │   ├── ln_1: LayerNorm (1,536 params)
│   │   │   ├── attn: GPT2Attention (2,362,368 params)
│   │   │   ├── ln_2: LayerNorm (1,536 params)
│   │   │   └── mlp: GPT2MLP (4,722,432 params)
│   │   ├── 7: GPT2Block (7,087,872 params)
│   │   │   ├── ln_1: LayerNorm (1,536 params)
│   │   │   ├── attn: GPT2Attention (2,362,368 params)
│   │   │   ├── ln_2: LayerNorm (1,536 params)
│   │   │   └── mlp: GPT2MLP (4,722,432 params)
│   │   ├── 8: GPT2Block (7,087,872 params)
│   │   │   ├── ln_1: LayerNorm (1,536 params)
│   │   │   ├── attn: GPT2Attention (2,362,368 params)
│   │   │   ├── ln_2: LayerNorm (1,536 params)
│   │   │   └── mlp: GPT2MLP (4,722,432 params)
│   │   ├── 9: GPT2Block (7,087,872 params)
│   │   │   ├── ln_1: LayerNorm (1,536 params)
│   │   │   ├── attn: GPT2Attention (2,362,368 params)
│   │   │   ├── ln_2: LayerNorm (1,536 params)
│   │   │   └── mlp: GPT2MLP (4,722,432 params)
│   │   ├── 10: GPT2Block (7,087,872 params)
│   │   │   ├── ln_1: LayerNorm (1,536 params)
│   │   │   ├── attn: GPT2Attention (2,362,368 params)
│   │   │   ├── ln_2: LayerNorm (1,536 params)
│   │   │   └── mlp: GPT2MLP (4,722,432 params)
│   │   └── 11: GPT2Block (7,087,872 params)
│   │       ├── ln_1: LayerNorm (1,536 params)
│   │       ├── attn: GPT2Attention (2,362,368 params)
│   │       ├── ln_2: LayerNorm (1,536 params)
│   │       └── mlp: GPT2MLP (4,722,432 params)
│   └── ln_f: LayerNorm (1,536 params)
└── lm_head: Linear (38,597,376 params)

TinyLlama Architecture (Total Params: 1,100,048,384)
├── model: LlamaModel (1,034,512,384 params)
│   ├── embed_tokens: Embedding (65,536,000 params)
│   ├── layers: ModuleList (968,974,336 params)
│   │   ├── 0: LlamaDecoderLayer (44,044,288 params)
│   │   │   ├── self_attn: LlamaAttention (9,437,184 params)
│   │   │   ├── mlp: LlamaMLP (34,603,008 params)
│   │   │   ├── input_layernorm: LlamaRMSNorm (2,048 params)
│   │   │   └── post_attention_layernorm: LlamaRMSNorm (2,048 params)
│   │   ├── 1: LlamaDecoderLayer (44,044,288 params)
│   │   │   ├── self_attn: LlamaAttention (9,437,184 params)
│   │   │   ├── mlp: LlamaMLP (34,603,008 params)
│   │   │   ├── input_layernorm: LlamaRMSNorm (2,048 params)
│   │   │   └── post_attention_layernorm: LlamaRMSNorm (2,048 params)
│   │   ├── 2: LlamaDecoderLayer (44,044,288 params)
│   │   │   ├── self_attn: LlamaAttention (9,437,184 params)
│   │   │   ├── mlp: LlamaMLP (34,603,008 params)
│   │   │   ├── input_layernorm: LlamaRMSNorm (2,048 params)
│   │   │   └── post_attention_layernorm: LlamaRMSNorm (2,048 params)
│   │   ├── 3: LlamaDecoderLayer (44,044,288 params)
│   │   │   ├── self_attn: LlamaAttention (9,437,184 params)
│   │   │   ├── mlp: LlamaMLP (34,603,008 params)
│   │   │   ├── input_layernorm: LlamaRMSNorm (2,048 params)
│   │   │   └── post_attention_layernorm: LlamaRMSNorm (2,048 params)
│   │   ├── 4: LlamaDecoderLayer (44,044,288 params)
│   │   │   ├── self_attn: LlamaAttention (9,437,184 params)
│   │   │   ├── mlp: LlamaMLP (34,603,008 params)
│   │   │   ├── input_layernorm: LlamaRMSNorm (2,048 params)
│   │   │   └── post_attention_layernorm: LlamaRMSNorm (2,048 params)
│   │   ├── 5: LlamaDecoderLayer (44,044,288 params)
│   │   │   ├── self_attn: LlamaAttention (9,437,184 params)
│   │   │   ├── mlp: LlamaMLP (34,603,008 params)
│   │   │   ├── input_layernorm: LlamaRMSNorm (2,048 params)
│   │   │   └── post_attention_layernorm: LlamaRMSNorm (2,048 params)
│   │   ├── 6: LlamaDecoderLayer (44,044,288 params)
│   │   │   ├── self_attn: LlamaAttention (9,437,184 params)
│   │   │   ├── mlp: LlamaMLP (34,603,008 params)
│   │   │   ├── input_layernorm: LlamaRMSNorm (2,048 params)
│   │   │   └── post_attention_layernorm: LlamaRMSNorm (2,048 params)
│   │   ├── 7: LlamaDecoderLayer (44,044,288 params)
│   │   │   ├── self_attn: LlamaAttention (9,437,184 params)
│   │   │   ├── mlp: LlamaMLP (34,603,008 params)
│   │   │   ├── input_layernorm: LlamaRMSNorm (2,048 params)
│   │   │   └── post_attention_layernorm: LlamaRMSNorm (2,048 params)
│   │   ├── 8: LlamaDecoderLayer (44,044,288 params)
│   │   │   ├── self_attn: LlamaAttention (9,437,184 params)
│   │   │   ├── mlp: LlamaMLP (34,603,008 params)
│   │   │   ├── input_layernorm: LlamaRMSNorm (2,048 params)
│   │   │   └── post_attention_layernorm: LlamaRMSNorm (2,048 params)
│   │   ├── 9: LlamaDecoderLayer (44,044,288 params)
│   │   │   ├── self_attn: LlamaAttention (9,437,184 params)
│   │   │   ├── mlp: LlamaMLP (34,603,008 params)
│   │   │   ├── input_layernorm: LlamaRMSNorm (2,048 params)
│   │   │   └── post_attention_layernorm: LlamaRMSNorm (2,048 params)
│   │   ├── 10: LlamaDecoderLayer (44,044,288 params)
│   │   │   ├── self_attn: LlamaAttention (9,437,184 params)
│   │   │   ├── mlp: LlamaMLP (34,603,008 params)
│   │   │   ├── input_layernorm: LlamaRMSNorm (2,048 params)
│   │   │   └── post_attention_layernorm: LlamaRMSNorm (2,048 params)
│   │   ├── 11: LlamaDecoderLayer (44,044,288 params)
│   │   │   ├── self_attn: LlamaAttention (9,437,184 params)
│   │   │   ├── mlp: LlamaMLP (34,603,008 params)
│   │   │   ├── input_layernorm: LlamaRMSNorm (2,048 params)
│   │   │   └── post_attention_layernorm: LlamaRMSNorm (2,048 params)
│   │   ├── 12: LlamaDecoderLayer (44,044,288 params)
│   │   │   ├── self_attn: LlamaAttention (9,437,184 params)
│   │   │   ├── mlp: LlamaMLP (34,603,008 params)
│   │   │   ├── input_layernorm: LlamaRMSNorm (2,048 params)
│   │   │   └── post_attention_lay

In [ ]:
#Task 2 - LoRA Fine-Tuning

In [3]:
# We use GPT-2 for fine-tuning as it fits perfectly in smaller GPUs
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token

print("Loading IMDB dataset subset...")
dataset = load_dataset("imdb", split="train[:500]") # Small subset for fast execution

def tokenize_function(examples):
    prompts = [f"Review: {text}\nSentiment: {'Positive' if label == 1 else 'Negative'}" 
               for text, label in zip(examples["text"], examples["label"])]
    return tokenizer(prompts, padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# LoRA Configuration targeting GPT-2's specific attention blocks
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8, 
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["c_attn", "c_proj"] 
)

peft_model = get_peft_model(gpt2_model, lora_config)
print("\nTrainable Parameters with LoRA:")
peft_model.print_trainable_parameters()
peft_model.to(device)

optimizer = torch.optim.AdamW(peft_model.parameters(), lr=2e-4)

print("\nStarting LoRA Fine-Tuning (1 Epoch)...")
peft_model.train()
start_time = time.time()

batch_size = 8 # Safe for your GPU
for i in range(0, len(tokenized_datasets), batch_size):
    batch = tokenized_datasets[i:i+batch_size]
    inputs = torch.tensor(batch['input_ids']).to(device)
    
    outputs = peft_model(inputs, labels=inputs)
    loss = outputs.loss
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    if i % 80 == 0:
        print(f"Step {i} | Loss: {loss.item():.4f}")

print(f"Training complete in {time.time() - start_time:.2f} seconds.")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading IMDB dataset subset...


...

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

c:\Users\Hardik\Desktop\SE\llm-systems-assessment\venv\Lib\site-packages\peft\tuners\lora\layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(



Trainable Parameters with LoRA:
trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475

Starting LoRA Fine-Tuning (1 Epoch)...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step 0 | Loss: 3.9428
Step 80 | Loss: 3.9931
Step 160 | Loss: 4.0659
Step 240 | Loss: 3.2879
Step 320 | Loss: 3.4305
Step 400 | Loss: 3.5992
Step 480 | Loss: 3.4012
Training complete in 23.45 seconds.


Task 3 - Model Composition (Before vs After)

In [4]:
def generate_text(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    # Using max_new_tokens to keep output concise
    outputs = model.generate(**inputs, max_new_tokens=15, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_prompt = "Review: The movie was incredibly boring and I hated the acting.\nSentiment:"

# Test the model with the LoRA adapter active
print("--- Output WITH LoRA Adapter (Before Merge) ---")
peft_model.eval()
print(generate_text(peft_model, test_prompt))

# MERGE the adapter back into the base model weights permanently
print("\nMerging LoRA weights into base model...")
merged_model = peft_model.merge_and_unload()

# Test the final, consolidated model
print("\n--- Output of MERGED Base Model ---")
merged_model.eval()
print(generate_text(merged_model, test_prompt))

print("\nTask 3 Complete! Adapter weights successfully averaged into base weights.")

--- Output WITH LoRA Adapter (Before Merge) ---
Review: The movie was incredibly boring and I hated the acting.
Sentiment: Very bad.
Rating: 4.5

Merging LoRA weights into base model...

--- Output of MERGED Base Model ---
Review: The movie was incredibly boring and I hated the acting.
Sentiment: Very bad.
Rating: 4.5

Task 3 Complete! Adapter weights successfully averaged into base weights.
